In [2]:
import pandas as pd
import numpy as np

# Load the training data split
df = pd.read_csv("data/train.csv.gz")

print(f"Shape: {df.shape}")
print("Column Info & Missing Values Inspection")
print(df.info())

print("First 5 Rows")
display(df.head())

print("Treatment Group vs Holdout Group")
# Treatment Group (treatment = 1): The audience chosen to receive the campaign. MarketBridge delivers SolePeak's ads to them.
# Holdout Group (treatment = 0): A randomly selected subset of users deliberately held back and blocked from seeing SolePeak's ads.
print(df['treatment'].value_counts(normalize=True))

print("Percentage of People Who Visited: Ad Group (1) vs Ad-Free Group (0)")
print(df.groupby('treatment')['visit'].mean())

Shape: (700000, 16)
Column Info & Missing Values Inspection
<class 'pandas.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 16 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   f0          700000 non-null  float64
 1   f1          700000 non-null  float64
 2   f2          700000 non-null  float64
 3   f3          700000 non-null  float64
 4   f4          700000 non-null  float64
 5   f5          700000 non-null  float64
 6   f6          700000 non-null  float64
 7   f7          700000 non-null  float64
 8   f8          700000 non-null  float64
 9   f9          700000 non-null  float64
 10  f10         700000 non-null  float64
 11  f11         700000 non-null  float64
 12  treatment   700000 non-null  int64  
 13  conversion  700000 non-null  int64  
 14  visit       700000 non-null  int64  
 15  exposure    700000 non-null  int64  
dtypes: float64(12), int64(4)
memory usage: 85.4 MB
None
First 5 Rows


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,18.255007,10.059654,8.873602,3.907662,10.280525,4.115453,-7.011752,4.833815,3.895862,13.190056,5.300375,-0.168679,1,0,0,0
1,25.315285,10.059654,8.214383,4.679882,10.280525,4.115453,-3.282109,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0
2,22.340863,10.059654,8.214383,4.679882,10.280525,4.115453,-10.006574,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0
3,20.831647,10.059654,8.944858,4.679882,11.561050,4.115453,-5.987667,4.833815,3.849228,13.190056,6.349310,-0.168679,1,0,0,0
4,12.616365,10.059654,8.985145,4.679882,10.280525,4.115453,0.294443,4.833815,3.934656,13.190056,5.300375,-0.168679,1,0,0,0


Treatment Group vs Holdout Group
treatment
1    0.849953
0    0.150047
Name: proportion, dtype: float64
Percentage of People Who Visited: Ad Group (1) vs Ad-Free Group (0)
treatment
0    0.038712
1    0.048391
Name: visit, dtype: float64


## 1. Initial Data Inspection & Baseline ITT Exploration (Train Split)

**Purpose:**
Establish baseline integrity for the training dataset (`train.csv.gz`), verify sample dimensions and completeness, and calculate the unadjusted Intention-to-Treat (ITT) baseline visit rate.

**Key Guardrails Applied:**
* **Feature Scope:** Predictor matrix is strictly limited to `f0`–`f11`. No engineered identities or invented labels.
* **Leakage Wall:** Diagnostic field `exposure` is strictly excluded from training features and filtering criteria.
* **Causal Protocol:** Evaluation follows the Intention-to-Treat (ITT) principle based on randomized `treatment` assignment, measuring the effect of ad eligibility rather than ad delivery.

**Takeaways & Key Insights:**
* **Pristine Data Hygiene:** Exactly 700,000 rows, 16 columns, and zero null values. No imputation headaches out of the gate.
* **The 85/15 Imbalance:** Treatment is heavily favored (85% ad-eligible vs. 15% holdout). While great for campaign reach, it means the control group is comparatively small (~105,000 users). Keep this sample size limitation in mind when slicing into granular cohorts to avoid overfitting to noisy holdouts.
* **The Benchmark to Beat:** 
  * Treated Visit Rate: **~4.84%**
  * Holdout Visit Rate: **~3.87%**
  * **Baseline ITT Lift: +0.97% percentage points.**
* **The Core Uplift Question:** Ads move the needle by roughly one percentage point overall. But are we uniformly nudging everyone, or is a small subset driving this entire lift while we waste ad spend on the rest? That's what uplift modeling needs to unpack.

In [3]:
# Load validation and test files alongside train
val_df = pd.read_csv("data/validation.csv.gz")
test_df = pd.read_csv("data/test.csv.gz")

splits = {
    'Train': df,
    'Validation': val_df,
    'Test': test_df
}

# Collect key metrics across files
summary_rows = []
for name, data in splits.items():
    n_total = len(data)
    n_treat = (data['treatment'] == 1).sum()
    n_ctrl = (data['treatment'] == 0).sum()
    treat_pct = (n_treat / n_total) * 100
    
    visit_rate_overall = data['visit'].mean() * 100
    visit_rate_treat = data.loc[data['treatment'] == 1, 'visit'].mean() * 100
    visit_rate_ctrl = data.loc[data['treatment'] == 0, 'visit'].mean() * 100
    itt_lift = visit_rate_treat - visit_rate_ctrl
    
    conv_rate_overall = data['conversion'].mean() * 100
    
    summary_rows.append({
        'Split': name,
        'Total Rows': n_total,
        'Treatment Share (%)': round(treat_pct, 2),
        'Overall Visit Rate (%)': round(visit_rate_overall, 3),
        'Treated Visit (%)': round(visit_rate_treat, 3),
        'Holdout Visit (%)': round(visit_rate_ctrl, 3),
        'ITT Visit Lift (% pts)': round(itt_lift, 3),
        'Overall Conversion Rate (%)': round(conv_rate_overall, 3)
    })

split_comparison = pd.DataFrame(summary_rows)
display(split_comparison)

,Split,Total Rows,Treatment Share (%),Overall Visit Rate (%),Treated Visit (%),Holdout Visit (%),ITT Visit Lift (% pts),Overall Conversion Rate (%)
0,Train,700000,85.0,4.694,4.839,3.871,0.968,0.284
1,Validation,150000,85.0,4.694,4.839,3.870,0.970,0.284
2,Test,150000,85.0,4.694,4.839,3.870,0.970,0.284


## 2. Dataset Split Consistency & Stability Audit

**Purpose:**
Validate partition integrity across Train (700k), Validation (150k), and Test (150k) splits to ensure the random assignment ratio (~85/15) and outcome baselines are stable without drift or sampling artifacts.

**Key Guardrails Applied:**
* **Leakage Prevention:** Metrics evaluated here are purely aggregate dataset-level diagnostics (`count`, `mean`). No features are processed, transformed, or evaluated on validation or test data.
* **Outcome Hierarchy:** Explicitly measures the extreme rarity of secondary `conversion` (~0.28%) against primary `visit` (~4.7%).
* **Evaluation Boundary:** `test.csv.gz` is audited solely for distribution balance and will remain strictly untouched for all modeling decisions until final evaluation.

**Takeaways & Key Insights:**
* **Flawless Split Proportions:** Total rows hit the exact 1,000,000 mark (700k / 150k / 150k), and the treatment share sits at an identical 85.00% across all three partitions. No sampling drift detected.
* **Solid Lift Stability:** The unadjusted ITT visit lift is consistent across partitions (+0.968% in Train vs. +0.970% in Validation and Test), confirming that the partitions represent identical underlying distributions.
* **Why Conversion is Strictly Secondary:** Overall conversion rates sit flat at a tiny ~0.284% across every partition. Conversions are roughly 16.5× rarer than visits (~4.69% vs. ~0.28%). Optimizing directly for conversions would introduce extreme noise; `visit` is our reliable North Star metric for ranking uplift.

In [4]:
# Isolate the permitted model features according to the data dictionary
feature_cols = [f'f{i}' for i in range(12)]

# Inspect basic descriptive statistics (min, max, quartiles) on Train only
train_features_summary = df[feature_cols].describe().T[['mean', 'std', 'min', '50%', 'max']]
train_features_summary.columns = ['Mean', 'Std Dev', 'Min', 'Median', 'Max']

print("Feature Matrix Summary (f0 - f11 on Train)")
display(train_features_summary.round(3))

Feature Matrix Summary (f0 - f11 on Train)


,Mean,Std Dev,Min,Median,Max
f0,19.616,5.378,12.616,21.921,26.745
f1,10.070,0.104,10.060,10.060,15.451
f2,8.446,0.299,8.214,8.214,9.052
f3,4.180,1.334,-6.554,4.680,4.680
f4,10.339,0.341,10.281,10.281,20.261
f5,4.029,0.427,-7.550,4.115,4.115
f6,-4.153,4.570,-29.220,-2.411,0.294
f7,5.101,1.205,4.834,4.834,11.998
f8,3.934,0.057,3.644,3.972,3.972
f9,16.036,7.030,13.190,13.190,68.701
